# Limpeza de Dados (Data Cleaning)
Neste notebook importamos os datasets reais `cobranca_assessorias.csv` e `fluxo_pagamentos.xlsx`, validamos encoding e tratamos nulos/inconsistências.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Caminhos
BASE_DIR = os.path.abspath('..')
cobranca_path = os.path.join(BASE_DIR, 'cobranca_assessorias.csv')
pagamentos_path = os.path.join(BASE_DIR, 'fluxo_pagamentos.xlsx')

# Leitura com tratamento de encoding
print("Lendo contratos...")
contratos = pd.read_csv(cobranca_path, encoding='latin1')
print(f"Contratos originais: {contratos.shape}")

print("Lendo pagamentos...")
pagamentos = pd.read_excel(pagamentos_path)
print(f"Pagamentos originais: {pagamentos.shape}")

Lendo contratos...
Contratos originais: (10000, 8)
Lendo pagamentos...


Pagamentos originais: (100000, 9)


### Limpeza e Padronização de Assessorias e Regiões

In [2]:
# Remover espaços e padronizar
contratos['Nome_Assessoria'] = contratos['Nome_Assessoria'].str.strip()
nome_lower = contratos['Nome_Assessoria'].str.lower()
contratos.loc[nome_lower.str.contains('rtice', na=False), 'Nome_Assessoria'] = 'Vertice Asset e Cobranca'
contratos.loc[nome_lower.str.contains('nix', na=False), 'Nome_Assessoria'] = 'Fenix Recuperacao de Credito'
contratos.loc[nome_lower.str.contains('nexus', na=False), 'Nome_Assessoria'] = 'Nexus Mediacao Financeira'
contratos.loc[nome_lower.str.contains('acerta', na=False), 'Nome_Assessoria'] = 'Acerta Credito Integrado'

contratos['Regiao_Cliente'] = contratos['Regiao_Cliente'].str.strip()
regiao_lower = contratos['Regiao_Cliente'].str.lower()
contratos.loc[regiao_lower == 'nordeste', 'Regiao_Cliente'] = 'Nordeste'
contratos.loc[regiao_lower == 'sudeste', 'Regiao_Cliente'] = 'Sudeste'
contratos.loc[regiao_lower == 'sul', 'Regiao_Cliente'] = 'Sul'
contratos.loc[regiao_lower.str.contains('centro', na=False), 'Regiao_Cliente'] = 'Centro-Oeste'
contratos.loc[regiao_lower == 'norte', 'Regiao_Cliente'] = 'Norte'

print("Assessorias únicas:", contratos['Nome_Assessoria'].unique())
print("Regiões únicas:", contratos['Regiao_Cliente'].unique())

Assessorias únicas: <StringArray>
[    'Vertice Asset e Cobranca',     'Acerta Credito Integrado',
    'Nexus Mediacao Financeira', 'Fenix Recuperacao de Credito']
Length: 4, dtype: str
Regiões únicas: <StringArray>
['Sudeste', 'Centro-Oeste', 'Nordeste', 'Sul', 'Norte']
Length: 5, dtype: str


### Parse de Valores Monetários e Datas

In [3]:
def parse_money(val):
    if pd.isna(val): return 0.0
    s = str(val).strip().replace('R$', '').strip()
    if ',' in s: s = s.replace('.', '').replace(',', '.')
    try: return float(s)
    except: return 0.0

contratos['Valor_Inadimplente_Inicial'] = contratos['Valor_Inadimplente_Inicial'].apply(parse_money)
contratos['Score_Interno_Risco'] = contratos['Score_Interno_Risco'].fillna(contratos['Score_Interno_Risco'].median())
contratos['Data_Envio_Assessoria'] = pd.to_datetime(contratos['Data_Envio_Assessoria'], errors='coerce')

pagamentos['Data_Vencimento'] = pd.to_datetime(pagamentos['Data_Vencimento'], errors='coerce')
pagamentos['Data_Pagamento'] = pd.to_datetime(pagamentos['Data_Pagamento'], errors='coerce')

print("Dados limpos com sucesso! Salvando versões clean...")
contratos.to_csv('contratos_clean.csv', index=False)
pagamentos.to_csv('pagamentos_clean.csv', index=False)

Dados limpos com sucesso! Salvando versões clean...
